# Build processed TEC dataset

This notebook downloads daily IONEX/GIM files into temporary Colab storage, parses them, aggregates TEC values by predefined regions, and saves processed regional time series as CSV and Parquet.

Raw IONEX files are not stored in Git or Google Drive.

In [ ]:
from pathlib import Path
import sys
import os
import shutil
import gzip
import re
from datetime import date, datetime, timedelta
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError

import numpy as np
import pandas as pd

In [ ]:
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # If the repo is cloned into /content/tec-agent-project
    PROJECT_ROOT = Path("/content/tec-agent-project")
    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/tec_agent_project")

else:
    # Local VS Code mode
    PROJECT_ROOT = Path.cwd()
    DRIVE_PROJECT_DIR = PROJECT_ROOT

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

TMP_DIR = Path("/content/tec_tmp") if IN_COLAB else PROJECT_ROOT / "data" / "cache" / "tec_tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR = DRIVE_PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TMP_DIR:", TMP_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

In [ ]:
from tec_agents.data.regions import REGIONS, list_region_ids

print("Regions:")
for region_id in list_region_ids():
    region = REGIONS[region_id]
    print(
        f"{region_id:24s} "
        f"lat=[{region.lat_min}, {region.lat_max}], "
        f"lon=[{region.lon_min}, {region.lon_max}]"
    )

In [ ]:
def daterange(start: date, end: date):
    """Yield dates in half-open interval [start, end)."""
    current = start
    while current < end:
        yield current
        current += timedelta(days=1)


def gps_week(d: date) -> int:
    """Return GPS week number for a date."""
    gps_epoch = date(1980, 1, 6)
    return (d - gps_epoch).days // 7


def doy(d: date) -> int:
    """Return day of year."""
    return d.timetuple().tm_yday

In [ ]:
def build_codg_url(d: date) -> str:
    """
    Build BKG IGS URL for CODE final GIM IONEX gz file.

    Example filename:
    COD0OPSFIN_20240610000_01D_01H_GIM.INX.gz
    """
    week = gps_week(d)
    year = d.year
    day = doy(d)

    filename = f"COD0OPSFIN_{year}{day:03d}0000_01D_01H_GIM.INX.gz"
    return f"https://igs.bkg.bund.de/root_ftp/IGS/products/{week}/{filename}"


def download_ionex_day(d: date, tmp_dir: Path = TMP_DIR, overwrite: bool = False) -> Path:
    """Download one daily IONEX gz file into temporary storage."""
    url = build_codg_url(d)
    out_path = tmp_dir / url.split("/")[-1]

    if out_path.exists() and not overwrite:
        return out_path

    print(f"Downloading {d} -> {out_path.name}")

    try:
        urlretrieve(url, out_path)
    except (HTTPError, URLError) as exc:
        raise RuntimeError(f"Failed to download {url}: {exc}") from exc

    return out_path

In [ ]:
_NUM_RE = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[DEde][-+]?\d+)?")


def _numbers_from_line(line: str) -> list[float]:
    txt = line[:80].replace("D", "E").replace("d", "E")
    return [float(x) for x in _NUM_RE.findall(txt)]


def _read_text_from_gz(path: Path) -> list[str]:
    with gzip.open(path, "rt", encoding="latin-1", errors="ignore") as f:
        return f.readlines()


def parse_ionex_gim(path: Path) -> pd.DataFrame:
    """
    Parse one daily IONEX GIM file into long DataFrame.

    Returns columns:
    time, lat, lon, tec

    Note:
    - TEC values in IONEX may be stored with exponent scaling.
    - This parser handles common IONEX map blocks and EXPONENT header.
    """
    lines = _read_text_from_gz(path)

    exponent = 0
    lat_grid = None
    lon_grid = None
    dlon = None

    # Header parse
    for line in lines:
        label = line[60:].strip() if len(line) >= 60 else ""

        if "EXPONENT" in label:
            nums = _numbers_from_line(line)
            if nums:
                exponent = int(nums[0])

        if "END OF HEADER" in line:
            break

    scale = 10 ** exponent

    records = []
    current_time = None
    in_map = False
    i = 0

    while i < len(lines):
        line = lines[i]
        label = line[60:].strip() if len(line) >= 60 else ""

        if "START OF TEC MAP" in label:
            in_map = True
            current_time = None
            i += 1
            continue

        if "END OF TEC MAP" in label:
            in_map = False
            i += 1
            continue

        if in_map and "EPOCH OF CURRENT MAP" in label:
            nums = _numbers_from_line(line)
            if len(nums) >= 6:
                y, mo, d, h, mi, s = [int(x) for x in nums[:6]]
                current_time = pd.Timestamp(datetime(y, mo, d, h, mi, s))
            i += 1
            continue

        if in_map and "LAT/LON1/LON2/DLON/H" in label:
            nums = _numbers_from_line(line)
            if len(nums) < 5:
                i += 1
                continue

            lat, lon1, lon2, dlon, height = nums[:5]
            lons = np.arange(lon1, lon2 + dlon / 2, dlon)

            values = []
            i += 1

            while i < len(lines):
                next_line = lines[i]
                next_label = next_line[60:].strip() if len(next_line) >= 60 else ""

                if (
                    "LAT/LON1/LON2/DLON/H" in next_label
                    or "END OF TEC MAP" in next_label
                    or "START OF TEC MAP" in next_label
                    or "EPOCH OF CURRENT MAP" in next_label
                ):
                    i -= 1
                    break

                values.extend(_numbers_from_line(next_line))
                if len(values) >= len(lons):
                    break

                i += 1

            values = values[: len(lons)]

            if current_time is not None and len(values) == len(lons):
                for lon, raw_value in zip(lons, values):
                    if raw_value >= 9999:
                        tec = np.nan
                    else:
                        tec = raw_value * scale

                    records.append(
                        {
                            "time": current_time,
                            "lat": float(lat),
                            "lon": float(lon),
                            "tec": float(tec) if np.isfinite(tec) else np.nan,
                        }
                    )

        i += 1

    df = pd.DataFrame.from_records(records)

    if df.empty:
        raise ValueError(f"No TEC records parsed from {path}")

    return df

In [ ]:
def normalize_lon(lon: pd.Series) -> pd.Series:
    """Normalize longitude to [-180, 180]."""
    return ((lon + 180) % 360) - 180


def aggregate_regions(df_long: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate long TEC table into regional mean time series.

    Input columns:
    time, lat, lon, tec

    Output:
    DatetimeIndex, one column per region_id.
    """
    df = df_long.copy()
    df["lon_norm"] = normalize_lon(df["lon"])

    result = {}

    for region_id, region in REGIONS.items():
        mask = (
            (df["lat"] >= region.lat_min)
            & (df["lat"] <= region.lat_max)
            & (df["lon_norm"] >= region.lon_min)
            & (df["lon_norm"] <= region.lon_max)
        )

        regional = (
            df.loc[mask]
            .groupby("time")["tec"]
            .mean()
            .sort_index()
        )

        result[region_id] = regional

    out = pd.DataFrame(result)
    out.index.name = "time"
    out = out.sort_index()

    return out

In [ ]:
START_DATE = date(2024, 3, 1)
END_DATE = date(2024, 4, 1)

OUTPUT_BASENAME = "tec_regions_2024_03_hourly"

print("Date range:", START_DATE, "to", END_DATE)
print("Output basename:", OUTPUT_BASENAME)

In [ ]:
daily_region_tables = []

for d in daterange(START_DATE, END_DATE):
    try:
        gz_path = download_ionex_day(d, TMP_DIR, overwrite=False)
        df_long = parse_ionex_gim(gz_path)
        df_regions_day = aggregate_regions(df_long)
        daily_region_tables.append(df_regions_day)

        print(
            f"{d}: parsed records={len(df_long):,}, "
            f"time_points={len(df_regions_day)}"
        )

    except Exception as exc:
        print(f"[ERROR] {d}: {exc}")

if not daily_region_tables:
    raise RuntimeError("No daily tables were built.")

df_regions = pd.concat(daily_region_tables).sort_index()
df_regions = df_regions[~df_regions.index.duplicated(keep="first")]

print("Final shape:", df_regions.shape)
df_regions.head()

In [ ]:
print("Time range:", df_regions.index.min(), "->", df_regions.index.max())
print("Columns:", list(df_regions.columns))
print("Missing values by column:")
print(df_regions.isna().sum())

display(df_regions.describe().T)

In [ ]:
import matplotlib.pyplot as plt

ax = df_regions[["midlat_europe", "highlat_north", "equatorial_atlantic"]].plot(
    figsize=(12, 4),
    grid=True,
    title="Regional mean TEC, March 2024",
)
ax.set_ylabel("TEC, TECU")
plt.show()

In [ ]:
csv_path = PROCESSED_DIR / f"{OUTPUT_BASENAME}.csv"
parquet_path = PROCESSED_DIR / f"{OUTPUT_BASENAME}.parquet"

df_regions.to_csv(csv_path, index=True)
df_regions.to_parquet(parquet_path)

print("Saved CSV:", csv_path)
print("Saved Parquet:", parquet_path)

print("CSV size MB:", csv_path.stat().st_size / 1024 / 1024)
print("Parquet size MB:", parquet_path.stat().st_size / 1024 / 1024)

In [ ]:
CLEAN_TMP = True

if CLEAN_TMP:
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    print("Temporary raw files removed:", TMP_DIR)
else:
    print("Temporary raw files kept:", TMP_DIR)

In [ ]:
from tec_agents.data.datasets import register_dataset, get_dataset_summary, get_region_series

register_dataset(
    dataset_ref="default",
    path=parquet_path,
    file_format="parquet",
)

summary = get_dataset_summary("default")
summary

In [ ]:
from tec_agents.tools.executor import build_default_executor

executor = build_default_executor(run_id="real_dataset_smoke")

ts_result = executor.call(
    "tec_get_timeseries",
    {
        "dataset_ref": "default",
        "region_id": "midlat_europe",
        "start": "2024-03-01",
        "end": "2024-04-01",
    },
)

threshold_result = executor.call(
    "tec_compute_high_threshold",
    {
        "series_id": ts_result["series_id"],
        "method": "quantile",
        "q": 0.9,
    },
)

intervals_result = executor.call(
    "tec_detect_high_intervals",
    {
        "series_id": ts_result["series_id"],
        "threshold_id": threshold_result["threshold_id"],
        "min_duration_minutes": 0,
        "merge_gap_minutes": 60,
    },
)

print("Threshold:", threshold_result)
print("Intervals:", intervals_result["n_intervals"])
intervals_result["intervals"][:5]